### mental health prediction - regression problem

In [5]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt
import seaborn as sns
from data_profiling import ProfileReport

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import (
    StandardScaler,
    OneHotEncoder,
    OrdinalEncoder,
    PowerTransformer
)

from sklearn.linear_model import LinearRegression , Ridge

from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor,
    HistGradientBoostingRegressor
)
from sklearn.tree import DecisionTreeRegressor
from xgboost import XGBRegressor

from sklearn.model_selection import KFold, cross_validate



from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score




In [6]:
df=pd.read_csv('Student Social Media And Mental Health Impact.csv')

In [7]:
df.sample(5)
df.shape

# (5000,13)
df.columns


Index(['Age', 'Gender', 'Country', 'Academic_Level', 'Most_Used_Platform',
       'Purpose_Of_Use', 'Avg_Daily_Usage_Hours', 'Daily_Unlocks',
       'Study_Hours', 'Physical_Activity_Hours', 'Sleep_Hours_Per_Night',
       'Stress_Level', 'Mental_Health_Score'],
      dtype='object')

In [8]:
# df.info()

# df.isnull().sum()

# df.describe()



### EDA

In [17]:
profile = ProfileReport(
    df,
    title="Student Mental Health EDA Report",
    explorative=True
)

profile.to_notebook_iframe()
# profile.to_file("mental_health_report.html")

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 14/14 [00:00<00:00, 583.49it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

### Feature Engineering
One meaningful engineered feature here: grouping Country.

Country has 111 unique values in this dataset — one-hot encoding that directly would add 110+ mostly-empty columns, which hurts the model far more than it helps (this is called high cardinality).
Dropping Country entirely throws away real signal — a student's country genuinely correlates with things like internet access, culture, and sleep norms.
The fix: keep the top 10 most frequent countries as their own category, and bucket everything else into "Other". We keep the signal that matters and lose the noise that doesn't.

In [10]:
### data cleaning


df = df.drop_duplicates()

#2. Converting negative or unrealitstic value to realistic value
df['Physical_Activity_Hours'] = df['Physical_Activity_Hours'].clip(lower=0)


### check skewness 

numerical_columns=df.select_dtypes('number')
numerical_columns.skew()

C:\Users\ASUS\AppData\Local\Temp\ipykernel_7856\19793402.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Physical_Activity_Hours'] = df['Physical_Activity_Hours'].clip(lower=0)


Age                        0.155008
Avg_Daily_Usage_Hours      0.005575
Daily_Unlocks              0.002309
Study_Hours                0.436125
Physical_Activity_Hours    0.053288
Sleep_Hours_Per_Night      0.123919
Mental_Health_Score        0.207086
dtype: float64

In [11]:
top_countries = df['Country'].value_counts().index[:10].tolist()

# top_countries


def group_countries(country):
  if country in top_countries:
    return country
  else:
    return 'Other'

In [12]:
df['Grouped_countries']=df['Country'].apply(group_countries)
df['Grouped_countries'].value_counts()

C:\Users\ASUS\AppData\Local\Temp\ipykernel_7856\1996513269.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Grouped_countries']=df['Country'].apply(group_countries)


Grouped_countries
Other        3231
India         389
USA           354
Canada        230
Australia     198
UK            185
Germany       136
Mexico         94
Turkey         94
France         87
Name: count, dtype: int64

In [13]:
numeric_cols = ["Age", "Avg_Daily_Usage_Hours", "Daily_Unlocks","Physical_Activity_Hours",
                      "Sleep_Hours_Per_Night","Study_Hours"]
ordinal_col        = ['Stress_Level']
normal_col         = ["Gender", "Academic_Level", "Most_Used_Platform", "Purpose_Of_Use", "Grouped_countries"]


feature_col =numeric_cols  + ordinal_col + normal_col
X = df[feature_col]
y = df['Mental_Health_Score']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=42)

In [14]:

numeric_transformation =Pipeline(steps=[
    ("imputer",SimpleImputer()),
    ("scaling",StandardScaler())
])

# df['Stress_Level'].value_counts()

ordinal_transformation=Pipeline(steps=[
    ("imputer",SimpleImputer(strategy="most_frequent")),
    ("encoding",OrdinalEncoder(categories=[['Low','Medium','High','Very High']]))
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])


preprocessor=ColumnTransformer(
    transformers=[
        ('numeric_transformation',numeric_transformation,numeric_cols),
        ('ordinal_transformation',ordinal_transformation,ordinal_col),
        ('categorical_transformer',categorical_transformer,normal_col)
    ],
    remainder="drop"
)



In [15]:
models = {
    "Linear Regression": LinearRegression(),
    "Ridge": Ridge(),
    "Random Forest": RandomForestRegressor(
        n_estimators=200, random_state=42
    ),
    "Gradient Boosting": GradientBoostingRegressor(
        random_state=42
    ),
    "XGBoost": XGBRegressor(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=4,
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1
    ),
    "decision_tree":DecisionTreeRegressor()
}

In [16]:
results=[]

test_results=[]


cv = KFold(n_splits=5, shuffle=True, random_state=42)

for name,regressor in models.items():
    pipe=Pipeline(steps=[
        ('preprocessing',preprocessor),
        ('regressor',regressor)
    ])
    scores=cross_validate(pipe,
        X_train,
        y_train,
        cv=cv,
        scoring={
            "mae": "neg_mean_absolute_error",
            "mse": "neg_mean_squared_error",
            "r2": "r2"
        },
        n_jobs=-1)

    # upar mae but after scores these scores add prefix as Test_name example - test_mae
    
    mae = -scores["test_mae"].mean()
    rmse = np.sqrt(-scores["test_mse"].mean())
    r2 = scores["test_r2"].mean()

    results.append({
        "Model": name,
        "CV_MAE": mae,
        "CV_RMSE": rmse,
        "CV_R2": r2})

    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)

    test_mae = mean_absolute_error(y_test, y_pred)
    test_mse = mean_squared_error(y_test, y_pred)
    test_rmse = np.sqrt(test_mse)
    test_r2 = r2_score(y_test, y_pred)
    
    test_results.append({
        "Model": name,
        "Test_MAE": test_mae,
        "Test_MSE": test_mse,
        "Test_RMSE": test_rmse,
        "Test_R2": test_r2
    })

results_df = pd.DataFrame(results).sort_values("CV_RMSE")
test_df = pd.DataFrame(test_results)
comparison = results_df.merge(test_df, on="Model")
print(comparison.to_string(index=False))

            Model   CV_MAE  CV_RMSE    CV_R2  Test_MAE  Test_MSE  Test_RMSE  Test_R2
    Random Forest 0.358479 0.485522 0.850857  0.345891  0.212964   0.461480 0.878748
          XGBoost 0.430745 0.557816 0.802967  0.439135  0.317765   0.563706 0.819080
Gradient Boosting 0.449305 0.579614 0.787310  0.465401  0.352920   0.594071 0.799063
    decision_tree 0.439136 0.663057 0.721306  0.424467  0.432580   0.657708 0.753709
            Ridge 0.522862 0.664178 0.720812  0.535659  0.456193   0.675421 0.740265
Linear Regression 0.522831 0.664192 0.720799  0.535766  0.456314   0.675511 0.740196
